### tokenize

```
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Chords \overline{AC} and \overline{DF} are equidistant ...
The final answer MUST BE put in \boxed{}.<|im_end|>
<|im_start|>assistant
<think>
```

- parquet images(bytes/path) -> PIL/structured chat message -> Qwen processor -> pixel_values + image_grid_thw + image placeholder token ids -> Qwen3.5 visual encoder -> image_embeds -> scatter 到 input_ids 中 image_token_id 对应的位置
- patch_size=16, spatial_merge_size=2, temporal_patch_size=2, in_channels=3, vision hidden=1024 → out 2560）
- input image.shape: (250, 258)
- encoding/embedding pipeline
    - 预处理 → patch 化，
        - processor 把图像缩放到 patch_size × merge_size = 16×2 = 32 的整数倍，得到约 256×256。
        - 切成 16×16 的 patch 网格 → image_grid_thw = [1, 16, 16]（t=1 帧, h=16, w=16）。
        - patch 数 = 1×16×16 = 256。每个 patch 展平成 temporal_patch_size×in_channels×patch_size² = 2×3×16×16 = 1536 维。
        - 即 pixel_values 形状 = (256, 1536)。
    - Vision tower（24 层自注意力，hidden=1024）
        - 对 256 个 patch token 做整图全自注意力，所以传给 flash 的 cu_seqlens=[0,256]、q_tokens=256 —— 这就是日志里前几次 q_tokens=256 → ok 的来源（视觉塔不传 mRoPE position_ids，所以 cu_seqlens 正确）。
        - 形状：(256, 1024)。
    - 2×2 空间合并 + 投影到语言维度
        - spatial_merge_size=2 → 256 / (2×2) = 64 个 token，投影到 out_hidden_size=2560。
        - 得到 image embeds (64, 2560)（即之前探针打印的 VISION pooled shape=(64,2560)）。
    - 注入语言序列
        - 文本 prompt 里的 `<image>` 被展开成 64 个 image_token_id=248056 占位 token；加上文本 token 和 chat template，整条 LLM 序列 = 121 个 token。
        - 用 masked_scatter 把那 64 个图像 embedding 填到 inputs_embeds 中占位 token 的位置上。
    - 语言模型层
        - 序列长 121 → full-attention 层调用 flash，本应 cu_seqlens=[0,121]、q_tokens=121。
        - mRoPE 的 position_ids 形状是 (3,1,121)：3 = (temporal, height, width) 三组旋转位置（模型已把第 0 行文本位置 text_position_ids 拆走，剩 3 行）；1 = batch；121 = 序列长。
        - bug 点：_is_packed_sequence 把这个 3D 张量误当成"3 条拼接序列"，于是 cu_seqlens = [0,121,242,363]，363 = 3×121，而真实只有 121 个 token → flash 越界。

-----

```
position_ids[:, 0, :] = text/global sequence position
position_ids[:, 1, :] = visual temporal position T
position_ids[:, 2, :] = visual height position H
position_ids[:, 3, :] = visual width position W
```

### RoPE -> MRoPE -> Interleaved MRoPE

- `Qwen3_5DecoderLayer` -> `Qwen3NextAttention` -> `get_rope(..., rope_parameters=config.rope_parameters)`
- Qwen3.5 用 3D position_ids（mRoPE，形状 `[3, bs, seq]）`。
- Qwen3.5 的 config.json
```
 "rope_parameters": {
   "mrope_interleaved": true,
   "mrope_section": [11, 11, 10],
   "rope_type": "default",
   "rope_theta": 10000000,
   "partial_rotary_factor": 0.25
 }
```

### verl

```python
from verl.utils import hf_processor, hf_tokenizer

# hf_processor
processor = AutoProcessor.from_pretrained(name_or_path, **kwargs)
# Qwen2VLProcessor, Qwen2_5_VLProcessor, Qwen3VLProcessor, 

# hf_tokenizer
tokenizer = AutoTokenizer.from_pretrained(name_or_path, **kwargs)
```